# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect the available record sets in this dataset.

We will reference all record sets and their fields by their `@id` as per Croissant best practices.

In [ ]:
# List all record set @ids and their field @ids from the metadata

record_sets = list(metadata.record_sets)
print(f"\nAvailable Record Sets and their Fields:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    field_ids = [f.id for f in rs.fields]
    print(f"    Field @ids: {field_ids}\n")

# If there are no RecordSets found, print a message (in some schemas, only one)
if not record_sets:
    print('No record sets found in the metadata. The dataset may consist of a single table.')
    # Try to show files attached
    if hasattr(metadata, 'files'):
        for f in metadata.files:
            print(f'File: {f.id}, name: {f.name}, encodingFormat: {getattr(f, "encoding_format", "unknown")}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, we'll display a small sample from each record set (if present).

In [ ]:
# Extract data from each record set into pandas DataFrames
record_sets = list(metadata.record_sets)
dataframes = {}

if record_sets:
    for rs in record_sets:
        print(f'Loading records for RecordSet @id: {rs.id}')
        try:
            records = list(dataset.records(record_set=rs.id))
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Fields: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"  Could not load records: {e}")
else:
    # If no explicit RecordSets present, load the main records
    print('Loading records from dataset (no explicit record sets)...')
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes['main'] = df
    print(f"Fields: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll identify available numeric fields and perform some filtering and normalization using their `@id` references.

In [ ]:
# Identify a numeric field from the first record set (or use a known field if available)
import numpy as np

# Use the first available DataFrame
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to auto-detect numeric columns
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to infer numeric fields if all are objects (common with CSVs)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    print(f'Numeric candidates: {numeric_candidates}')
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize field
        norm_col = f'{numeric_field}_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try group-by a candidate for grouping (choose a non-numeric column)
        obj_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = obj_candidates[0] if obj_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable grouping field found.')
    else:
        print("No numeric field available for analysis.")
else:
    print('No dataframes available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

To illustrate, we'll plot the distribution of the selected numeric field, and if there is a grouping field, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No available numeric field to plot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and inspected the metadata and records from the dataset using their Croissant schema.
- Record sets and fields are accessed by their `@id` for clarity and reproducibility.
- Basic EDA demonstrated numeric filtering, normalization, and grouping.
- Visualizations highlighted the distribution and potential group differences in the numeric field.

You may further customize this notebook to perform advanced analysis specific to your project.